In [1]:
import pandas

In [2]:
csv = pandas.read_csv('spark.csv')

In [3]:
#all_kv = {}
full_kv = {}
for code in csv['fcss_data']:
    kv = {}
    codes = code.strip().split(" ")
    for c in codes:
        if c in kv:
            kv[c] += 1
        else:
            kv[c] = 1
    for k, v in kv.items():
        if k in full_kv:
            full_kv[k] = max(v, full_kv[k])
        else:
            full_kv[k] = v
print(full_kv)
csv['fcss_data']

{'6,06': 3, '3300751': 9, '7563750': 18, '7564751': 4, '6,06N1N2N3': 1, '66,10': 1, '7566751': 4, '7565750': 4, '5,02': 1, '6,00': 1, '6,02': 1, '66,02': 1, '0200461': 2, '0202131': 1, '0202411': 1, '0203411': 1, '0301330': 1, '0301411': 1, '1300461': 1, '1303411': 2, '3302410': 4, '4101461': 2, '4105411': 1, '4600461': 3, '3301411': 3, '4162751': 6, '4164751': 3, '4163410': 6, '0200331': 4, '0262751': 6, '0264751': 3, '0263020': 6, '0200131': 2, '0201131': 2, '0202021': 1, '1300131': 1, '6,06N1': 1, '0200351': 2, '3600751': 2, '6,06N1S2': 1, '66,10N1S4': 2, '66B6,14N6S13': 1, '2400251': 2, '2500331': 2, '2500341': 2, '2500371': 2, '6,08N1Q2': 1, '66,00N1Q4': 2, '66B6,00N6Q13': 1, '0262121': 1, '1200331': 2, '1263750': 4, '1100131': 1, '1100331': 1, '1101331': 1, '1162131': 1, '1162441': 1, '1300331': 1, '4103411': 2, '0302110': 3, '0303760': 1, '1102760': 1, '3301330': 1, '3302330': 1, '3302460': 1, '3304750': 2, '3306750': 1, '4600751': 3, '7503751': 3}


0                  6,06 3300751 3300751 7563750 7563750
1                  6,06 3300751 3300751 7564751 7564751
2     6,06 3300751 3300751 3300751 7563750 7563750 7...
3     6,06 6,06 3300751 3300751 3300751 3300751 3300...
4     6,06 6,06 6,06 3300751 3300751 3300751 3300751...
5     6,06 6,06 6,06 6,06N1N2N3 3300751 3300751 3300...
6       6,06 6,06 66,10 3300751 3300751 7566751 7566751
7       6,06 6,06 66,10 3300751 3300751 7565750 7565750
8     6,06 6,06 66,10 3300751 3300751 3300751 756475...
9     6,06 6,06 66,10 3300751 3300751 3300751 330075...
10    5,02 6,00 6,02 66,02 0200461 0200461 0202131 0...
11    6,06 3300751 3300751 3300751 3301411 4162751 4...
12    6,06 3300751 3300751 3300751 3301411 3301411 4...
13    6,06 3300751 3300751 3300751 3301411 3301411 3...
14    6,06 0200331 0262751 0262751 0264751 3300751 3...
15    6,06 0200331 0200331 0262751 0262751 0262751 0...
16    6,06 0200331 0200331 0200331 0262751 0262751 0...
17    6,06 6,06 0200331 0200331 0262751 0262751 

In [4]:
keys = sorted(list(full_kv.keys()))
cumulative_keys = {}
cumulative_values = {}
cumulative_sum = 0
for k in keys:
    cumulative_keys[k] = cumulative_sum
    for v in range(0, full_kv[k]):
        cumulative_values[cumulative_sum + v] = k
    cumulative_sum += full_kv[k]


In [5]:
def encode_fcsp(code):
    words = code.strip().split(" ")
    freq = {}
    fimi = []
    for w in words:
        if w in freq:
            freq[w] += 1
        else:
            freq[w] = 1
        fimi.append(cumulative_keys[w] + freq[w])
    fimi = map(lambda x: str(x), sorted(fimi))
    return " ".join(fimi)

        

In [6]:
def decode_fcsp(fimi_line):
    fimi = [int(w) for w in fimi_line.strip().split(" ")]
    words = [cumulative_values[f] for f in fimi]
    return " ".join(sorted(words))
    

In [7]:
def normalize(fcsp):
    return " ".join(sorted(fcsp.split(" ")))

In [8]:
normalize(csv['fcss_data'][11])

'3300751 3300751 3300751 3301411 4162751 4162751 4164751 6,06 7563750 7563750 7563750 7563750 7563750 7563750'

In [9]:
fimi_first = encode_fcsp(csv['fcss_data'][11])
fimi_first

'64 65 66 74 93 94 105 117 135 136 137 138 139 140'

In [10]:
rtt_first = decode_fcsp(fimi_first)
rtt_first

'3300751 3300751 3300751 3301411 4162751 4162751 4164751 6,06 7563750 7563750 7563750 7563750 7563750 7563750'

In [11]:
def create_value_range_encoder(fimi_base, values, levels):
    step = (max(values) - min(values)) / levels
    m = min(values)
    ma = max(values)
    print(max(values), m)
    def encode(value):
        if (value < m):
            return fimi_base
        elif value > ma:
            return fimi_base + levels
        else:
            return fimi_base + round((value-m) / step)
    return encode

In [14]:
def create_value_range_encoder(fimi_base, values, levels):
    step = (max(values) - min(values)) / levels
    m = min(values)
    ma = max(values)
    print(ma, m)
    def encode(value):
        if (value < m):
            return str(fimi_base)
        elif value > ma:
            return str(fimi_base+levels)
        else:
            x = fimi_base + round((value-m) / step)
            if x == fimi_base:
                return " ".join([str(y) for y in [x, x+1]])
            elif x == fimi_base + levels:
                return " ".join([str(y[x-1, x])
            return " ".join([x-1, x, x+1])
    return encode

In [15]:
max_nmr_level = 5
max_nmr_encoder = create_value_range_encoder(cumulative_sum, csv['Max NMR Shift (ppm)'], max_nmr_level)
max_fimi = cumulative_sum + max_nmr_level + 1

170.2 141.0


In [16]:
min_nmr_level = 5
min_nmr_encoder = create_value_range_encoder(max_fimi, csv['Min NMR Shift (ppm)'], min_nmr_level)
max_fimi = max_fimi + min_nmr_level + 1

128.1 12.6


In [17]:
max_nmr_encoder(190), min_nmr_encoder(100)

TypeError: sequence item 0: expected str instance, int found

In [54]:
max_fimi

174

In [62]:
def create_binary_prop_encoder(threshold, high_word, low_word):
    def encoder(value):
        return low_word if value < threshold else high_word
    return encoder

In [63]:
prop_encoder = create_binary_prop_encoder(9.0, ">=9", "<9")

In [64]:
with open("spark.fimi", "w") as f:
    f.write("# attributes: %d properties: B(>=9,<9)\n" % (max_fimi+1))
    for i in range(0, len(csv)):
        fcsp_line = encode_fcsp(csv['fcss_data'][i])
        max_nmr_line = max_nmr_encoder(csv['Max NMR Shift (ppm)'][i])
        min_nmr_line = min_nmr_encoder(csv['Min NMR Shift (ppm)'][i])
        full_line = fcsp_line + " %s %s" % (max_nmr_line, min_nmr_line)
        prop_part = " | %s" % prop_encoder(csv['Spark Energy (J)'][i])
        f.write(full_line+prop_part+"\n")